# Assignment: Building a Modular Data Sanitization & Exploration Engine

## Solution Notebook

This notebook implements the full `DataInspector` and `PlottingMethods` classes, followed by a complete demonstration using the Titanic dataset.

## Section 1 — Install & Import Dependencies

In [ ]:
# Install required libraries (run once in Colab)
!pip install plotly scikit-learn scipy pandas numpy -q

In [ ]:
import io
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import pointbiserialr, f_oneway
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from itertools import combinations

warnings.filterwarnings('ignore')

# Colab file upload utility
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('All dependencies loaded successfully.')

## Section 2 — The `PlottingMethods` Class

In [ ]:
class PlottingMethods:
    """
    A modular, reusable class for generating individual Plotly charts.
    Each method returns an HTML-wrapped Plotly figure for flexible embedding.
    """

    @staticmethod
    def bar_chart(data: pd.Series, title: str = 'Bar Chart',
                  x_label: str = 'Category', y_label: str = 'Count',
                  color: str = '#636EFA') -> str:
        """
        Generate an interactive bar chart from a pandas Series.

        Parameters
        ----------
        data : pd.Series
            Series where the index is categories and values are counts/frequencies.
        title : str
            Chart title.
        x_label : str
            Label for the x-axis.
        y_label : str
            Label for the y-axis.
        color : str
            Bar fill color (hex or named).

        Returns
        -------
        str
            HTML string containing the interactive figure.
        """
        if data is None or data.empty:
            return '<p>No data provided for bar chart.</p>'

        fig = go.Figure(
            go.Bar(
                x=data.index.astype(str),
                y=data.values,
                marker_color=color,
                text=[f'{v}' for v in data.values],
                textposition='outside'
            )
        )
        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title=y_label,
            template='plotly_white',
            height=450
        )
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    @staticmethod
    def pie_chart(data: pd.Series, title: str = 'Pie Chart') -> str:
        """
        Generate an interactive pie chart from a pandas Series.

        Parameters
        ----------
        data : pd.Series
            Series where the index is labels and values are sizes.
        title : str
            Chart title.

        Returns
        -------
        str
            HTML string containing the interactive figure.
        """
        if data is None or data.empty:
            return '<p>No data provided for pie chart.</p>'

        fig = go.Figure(
            go.Pie(
                labels=data.index.astype(str),
                values=data.values,
                hole=0.3,
                textinfo='label+percent'
            )
        )
        fig.update_layout(
            title=title,
            template='plotly_white',
            height=450
        )
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    @staticmethod
    def histogram(series: pd.Series, title: str = 'Histogram',
                  x_label: str = 'Value', bins: int = 30,
                  color: str = '#EF553B') -> str:
        """
        Generate an interactive histogram from a numeric pandas Series.

        Parameters
        ----------
        series : pd.Series
            Numeric data series.
        title : str
            Chart title.
        x_label : str
            Label for the x-axis.
        bins : int
            Number of histogram bins.
        color : str
            Bar fill color.

        Returns
        -------
        str
            HTML string containing the interactive figure.
        """
        if series is None or series.dropna().empty:
            return '<p>No data provided for histogram.</p>'

        fig = go.Figure(
            go.Histogram(
                x=series.dropna(),
                nbinsx=bins,
                marker_color=color,
                opacity=0.85
            )
        )
        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title='Frequency',
            template='plotly_white',
            height=450
        )
        return fig.to_html(full_html=False, include_plotlyjs='cdn')


print('PlottingMethods class defined.')

## Section 3 — The `DataInspector` Class

In [ ]:
class DataInspector:
    """
    A comprehensive, reusable data sanitization and exploration engine.

    Capabilities
    ------------
    - CSV ingestion with garbage-string handling and auto-type correction
    - Structural analysis: shape, dtypes, missing values
    - Intelligent imputation (mean / median / mode / constant)
    - Duplicate & IQR-based outlier management
    - Feature engineering: numeric scaling & categorical encoding
    - Advanced interactive Plotly visualizations
    - Unified association heatmap (Pearson, Cramér's V, Eta / Point-Biserial)
    """

    # Strings that should be treated as NaN during ingestion
    GARBAGE_STRINGS = ['?', 'n/a', 'N/A', 'na', 'NA', 'NULL', 'null',
                        'None', 'none', 'NONE', ' ', '', '--', 'nan', 'NaN']

    # ------------------------------------------------------------------ #
    #  1. DATA INGESTION & SANITIZATION                                   #
    # ------------------------------------------------------------------ #

    def __init__(self):
        """Initialise the DataInspector with an empty DataFrame."""
        self.df: pd.DataFrame = pd.DataFrame()

    def upload_data(self) -> None:
        """
        Upload a CSV file interactively in Google Colab or via a file path
        outside Colab, then sanitize the raw data.

        In Colab: opens the native file-picker dialog.
        Outside Colab: prompts the user to type a local file path.
        """
        if IN_COLAB:
            uploaded = colab_files.upload()
            if not uploaded:
                print('No file uploaded.')
                return
            filename = list(uploaded.keys())[0]
            raw_df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                na_values=self.GARBAGE_STRINGS,
                keep_default_na=True
            )
        else:
            path = input('Enter the CSV file path: ').strip()
            raw_df = pd.read_csv(
                path,
                na_values=self.GARBAGE_STRINGS,
                keep_default_na=True
            )

        self.df = self._auto_type_correction(raw_df)
        print(f'Data loaded successfully: {self.df.shape[0]} rows × {self.df.shape[1]} columns.')

    def load_from_dataframe(self, df: pd.DataFrame) -> None:
        """
        Load data directly from an existing pandas DataFrame (useful for
        testing outside the Colab file-picker workflow).

        Parameters
        ----------
        df : pd.DataFrame
            The DataFrame to ingest and sanitize.
        """
        if df is None or df.empty:
            print('Provided DataFrame is empty or None.')
            return
        # Replace garbage strings that may have slipped in
        raw_df = df.replace(self.GARBAGE_STRINGS, np.nan)
        self.df = self._auto_type_correction(raw_df)
        print(f'DataFrame loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns.')

    def load_from_url(self, url: str) -> None:
        """
        Load a CSV directly from a URL.

        Parameters
        ----------
        url : str
            Public URL pointing to a CSV file.
        """
        raw_df = pd.read_csv(
            url,
            na_values=self.GARBAGE_STRINGS,
            keep_default_na=True
        )
        self.df = self._auto_type_correction(raw_df)
        print(f'Data loaded from URL: {self.df.shape[0]} rows × {self.df.shape[1]} columns.')

    def _auto_type_correction(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Attempt to coerce object columns to numeric types.
        A column is converted only if the result is NOT entirely NaN,
        preserving columns that are genuinely categorical.

        Parameters
        ----------
        df : pd.DataFrame
            Raw DataFrame post garbage-string replacement.

        Returns
        -------
        pd.DataFrame
            DataFrame with corrected dtypes.
        """
        df = df.copy()
        for col in df.columns:
            if df[col].dtype == object:
                converted = pd.to_numeric(df[col], errors='coerce')
                # Only apply conversion if at least one value survived
                if converted.notna().any():
                    df[col] = converted
        return df

    # ------------------------------------------------------------------ #
    #  2. STRUCTURAL ANALYSIS & CLEANING                                  #
    # ------------------------------------------------------------------ #

    def _check_data(self) -> bool:
        """Guard: return False and print a warning if df is empty."""
        if self.df.empty:
            print('No data loaded. Call upload_data() or load_from_dataframe() first.')
            return False
        return True

    @property
    def numeric_columns(self) -> list:
        """List of numeric column names in the current DataFrame."""
        return self.df.select_dtypes(include=[np.number]).columns.tolist()

    @property
    def categorical_columns(self) -> list:
        """List of non-numeric (categorical) column names."""
        return self.df.select_dtypes(exclude=[np.number]).columns.tolist()

    def data_summary(self) -> None:
        """
        Display a comprehensive structural summary of the dataset:
        - Shape (rows × columns)
        - First 20 rows preview
        - Per-column dtype, non-null count, and missing value percentage
        - Breakdown of numeric vs. categorical columns
        - Descriptive statistics for numeric columns
        """
        if not self._check_data():
            return

        print('=' * 60)
        print(f'DATASET SHAPE: {self.df.shape[0]} rows × {self.df.shape[1]} columns')
        print('=' * 60)

        print('\n--- First 20 Rows Preview ---')
        display(self.df.head(20))

        print('\n--- Column Info ---')
        info_df = pd.DataFrame({
            'dtype': self.df.dtypes,
            'non_null_count': self.df.notna().sum(),
            'null_count': self.df.isna().sum(),
            'null_%': (self.df.isna().sum() / len(self.df) * 100).round(2)
        })
        display(info_df)

        print(f'\n--- Column Breakdown ---')
        print(f'Numeric  columns ({len(self.numeric_columns)}): {self.numeric_columns}')
        print(f'Categorical columns ({len(self.categorical_columns)}): {self.categorical_columns}')

        print('\n--- Descriptive Statistics (Numeric) ---')
        display(self.df[self.numeric_columns].describe().T)

    def handle_missing_values(self, strategy: str = 'mean',
                               constant_value=0,
                               columns: list = None) -> None:
        """
        Impute missing values using the chosen strategy.

        Parameters
        ----------
        strategy : str
            One of 'mean', 'median', 'mode', or 'constant'.
        constant_value : scalar
            Value used when strategy='constant'.
        columns : list, optional
            Subset of columns to impute; defaults to all columns.
        """
        if not self._check_data():
            return

        cols = columns if columns else self.df.columns.tolist()
        valid_strategies = {'mean', 'median', 'mode', 'constant'}
        if strategy not in valid_strategies:
            print(f'Invalid strategy. Choose from: {valid_strategies}')
            return

        filled = 0
        for col in cols:
            before = self.df[col].isna().sum()
            if before == 0:
                continue

            if strategy == 'mean':
                if col in self.numeric_columns:
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
            elif strategy == 'median':
                if col in self.numeric_columns:
                    self.df[col].fillna(self.df[col].median(), inplace=True)
            elif strategy == 'mode':
                mode_val = self.df[col].mode()
                if not mode_val.empty:
                    self.df[col].fillna(mode_val[0], inplace=True)
            elif strategy == 'constant':
                self.df[col].fillna(constant_value, inplace=True)

            after = self.df[col].isna().sum()
            filled += (before - after)

        print(f'Imputation complete ({strategy}). Total cells filled: {filled}.')

    def remove_duplicates(self) -> None:
        """
        Remove exact duplicate rows from the DataFrame and report how
        many rows were dropped.
        """
        if not self._check_data():
            return
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        removed = before - len(self.df)
        print(f'Duplicate removal: {removed} row(s) dropped. Remaining: {len(self.df)}.')

    def handle_outliers(self, columns: list = None,
                         action: str = 'flag',
                         iqr_multiplier: float = 1.5) -> pd.DataFrame:
        """
        Detect and optionally remove IQR-based outliers.

        Parameters
        ----------
        columns : list, optional
            Numeric columns to inspect; defaults to all numeric columns.
        action : str
            'flag'   — add a boolean column '<col>_outlier' and return summary.
            'remove' — drop rows where any specified column is an outlier.
        iqr_multiplier : float
            Fence multiplier (default 1.5; use 3.0 for extreme outliers).

        Returns
        -------
        pd.DataFrame
            Summary table of outlier counts per column.
        """
        if not self._check_data():
            return pd.DataFrame()

        cols = columns if columns else self.numeric_columns
        if not cols:
            print('No numeric columns available for outlier detection.')
            return pd.DataFrame()

        outlier_masks = {}
        summary_rows = []

        for col in cols:
            if col not in self.df.columns:
                print(f'Column "{col}" not found. Skipping.')
                continue
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - iqr_multiplier * iqr
            upper = q3 + iqr_multiplier * iqr
            mask = (self.df[col] < lower) | (self.df[col] > upper)
            outlier_masks[col] = mask

            summary_rows.append({
                'column': col,
                'Q1': round(q1, 4),
                'Q3': round(q3, 4),
                'IQR': round(iqr, 4),
                'lower_fence': round(lower, 4),
                'upper_fence': round(upper, 4),
                'outlier_count': int(mask.sum()),
                'outlier_%': round(mask.sum() / len(self.df) * 100, 2)
            })

            if action == 'flag':
                self.df[f'{col}_outlier'] = mask

        summary_df = pd.DataFrame(summary_rows)

        if action == 'remove' and outlier_masks:
            combined_mask = pd.concat(outlier_masks, axis=1).any(axis=1)
            before = len(self.df)
            self.df = self.df[~combined_mask].reset_index(drop=True)
            print(f'Outlier removal: {before - len(self.df)} row(s) dropped. Remaining: {len(self.df)}.')

        display(summary_df)
        return summary_df

    def delete_rows(self, indices: str = None) -> None:
        """
        Interactively delete rows by index.

        Parameters
        ----------
        indices : str, optional
            Comma-separated string of integer row indices (e.g. '0,5,12').
            If None, the user is prompted interactively.
        """
        if not self._check_data():
            return
        if indices is None:
            indices = input('Enter comma-separated row indices to delete: ').strip()
        try:
            idx_list = [int(i.strip()) for i in indices.split(',') if i.strip()]
            valid = [i for i in idx_list if i in self.df.index]
            invalid = set(idx_list) - set(valid)
            if invalid:
                print(f'Warning: indices not found and ignored: {invalid}')
            self.df.drop(index=valid, inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            print(f'Deleted {len(valid)} row(s). Remaining: {len(self.df)}.')
        except ValueError:
            print('Invalid input. Please enter integers separated by commas.')

    def delete_columns(self, columns: str = None) -> None:
        """
        Interactively delete columns by name.

        Parameters
        ----------
        columns : str, optional
            Comma-separated column names (e.g. 'Age,Cabin,Ticket').
            If None, the user is prompted interactively.
        """
        if not self._check_data():
            return
        if columns is None:
            columns = input('Enter comma-separated column names to delete: ').strip()
        col_list = [c.strip() for c in columns.split(',') if c.strip()]
        valid = [c for c in col_list if c in self.df.columns]
        invalid = set(col_list) - set(valid)
        if invalid:
            print(f'Warning: columns not found and ignored: {invalid}')
        self.df.drop(columns=valid, inplace=True)
        print(f'Deleted columns: {valid}. Remaining columns: {self.df.columns.tolist()}.')

    # ------------------------------------------------------------------ #
    #  3. FEATURE ENGINEERING PREPARATION (NORMALISATION)                 #
    # ------------------------------------------------------------------ #

    def extract_normalized_numeric_data(self,
                                         method: str = 'minmax',
                                         columns: list = None) -> pd.DataFrame:
        """
        Scale numeric columns and return the result as a new DataFrame.

        Parameters
        ----------
        method : str
            'minmax'   — scales each column to [0, 1].
            'standard' — Z-score standardisation (mean=0, std=1).
            'robust'   — IQR-based scaling (robust to outliers).
        columns : list, optional
            Subset of numeric columns to scale; defaults to all.

        Returns
        -------
        pd.DataFrame
            DataFrame of scaled numeric features.
        """
        if not self._check_data():
            return pd.DataFrame()

        cols = columns if columns else self.numeric_columns
        # Keep only columns that exist and are truly numeric
        cols = [c for c in cols if c in self.df.columns
                and pd.api.types.is_numeric_dtype(self.df[c])]
        if not cols:
            print('No valid numeric columns found.')
            return pd.DataFrame()

        scalers = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }
        if method not in scalers:
            print(f'Invalid method. Choose from: {list(scalers.keys())}')
            return pd.DataFrame()

        subset = self.df[cols].copy()
        # Fill NaNs temporarily so scaler does not raise errors
        subset.fillna(subset.median(), inplace=True)
        scaled_array = scalers[method].fit_transform(subset)
        scaled_df = pd.DataFrame(scaled_array,
                                  columns=[f'{c}_scaled' for c in cols],
                                  index=self.df.index)
        print(f'Numeric scaling ({method}) applied to {len(cols)} column(s).')
        return scaled_df

    def extract_normalized_categorical_data(self,
                                             method: str = 'onehot',
                                             columns: list = None) -> pd.DataFrame:
        """
        Encode categorical columns and return the result as a new DataFrame.

        Parameters
        ----------
        method : str
            'onehot'  — binary dummy columns for each category.
            'ordinal' — integer codes (0, 1, 2 …) per category.
            'uniform' — like ordinal but then scaled to [0, 1].
        columns : list, optional
            Subset of categorical columns to encode; defaults to all.

        Returns
        -------
        pd.DataFrame
            DataFrame of encoded categorical features.
        """
        if not self._check_data():
            return pd.DataFrame()

        cols = columns if columns else self.categorical_columns
        cols = [c for c in cols if c in self.df.columns]
        if not cols:
            print('No categorical columns found.')
            return pd.DataFrame()

        subset = self.df[cols].copy()
        # Fill NaN with the mode so encoders don't fail
        for c in cols:
            mode_val = subset[c].mode()
            if not mode_val.empty:
                subset[c].fillna(mode_val[0], inplace=True)
            else:
                subset[c].fillna('Unknown', inplace=True)

        if method == 'onehot':
            enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = enc.fit_transform(subset)
            feature_names = enc.get_feature_names_out(cols)
            result = pd.DataFrame(encoded, columns=feature_names, index=self.df.index)

        elif method in ('ordinal', 'uniform'):
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded = enc.fit_transform(subset)
            result = pd.DataFrame(encoded,
                                   columns=[f'{c}_ordinal' for c in cols],
                                   index=self.df.index)
            if method == 'uniform':
                scaler = MinMaxScaler()
                result = pd.DataFrame(
                    scaler.fit_transform(result),
                    columns=[f'{c}_uniform' for c in cols],
                    index=self.df.index
                )
        else:
            print(f'Invalid method. Choose from: onehot, ordinal, uniform')
            return pd.DataFrame()

        print(f'Categorical encoding ({method}) applied to {len(cols)} column(s).')
        return result

    def get_merged_normalized_dataset(self,
                                       numeric_method: str = 'minmax',
                                       categorical_method: str = 'onehot') -> pd.DataFrame:
        """
        Merge scaled numeric features with encoded categorical features
        into a single model-ready DataFrame.

        Parameters
        ----------
        numeric_method : str
            Scaling method passed to extract_normalized_numeric_data().
        categorical_method : str
            Encoding method passed to extract_normalized_categorical_data().

        Returns
        -------
        pd.DataFrame
            Unified, fully normalised DataFrame.
        """
        num_df = self.extract_normalized_numeric_data(method=numeric_method)
        cat_df = self.extract_normalized_categorical_data(method=categorical_method)

        parts = [p for p in [num_df, cat_df] if not p.empty]
        if not parts:
            print('No data to merge.')
            return pd.DataFrame()

        merged = pd.concat(parts, axis=1)
        print(f'Merged dataset shape: {merged.shape[0]} rows × {merged.shape[1]} columns.')
        return merged

    # ------------------------------------------------------------------ #
    #  4. ADVANCED INTERACTIVE VISUALISATION (PLOTLY)                     #
    # ------------------------------------------------------------------ #

    def plot_univariate(self, column: str) -> None:
        """
        Render a 3-panel Plotly subplot for a single numeric column:
        (1) Horizontal Violin + Box overlay
        (2) Scatter plot (Index vs Value)
        (3) Histogram with KDE-like rug

        Parameters
        ----------
        column : str
            Name of a numeric column in the DataFrame.
        """
        if not self._check_data():
            return
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return
        if column not in self.numeric_columns:
            print(f'Column "{column}" is not numeric.')
            return

        series = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=(
                f'{column} — Violin/Box',
                f'{column} — Scatter (Index vs Value)',
                f'{column} — Histogram'
            )
        )

        # Panel 1: Violin + Box
        fig.add_trace(
            go.Violin(
                x=series, name=column,
                box_visible=True, meanline_visible=True,
                fillcolor='#636EFA', opacity=0.6,
                line_color='#1f2d7d', orientation='h'
            ),
            row=1, col=1
        )

        # Panel 2: Scatter
        fig.add_trace(
            go.Scatter(
                x=series.index, y=series,
                mode='markers',
                marker=dict(color='#EF553B', size=4, opacity=0.6),
                name=column
            ),
            row=1, col=2
        )

        # Panel 3: Histogram
        fig.add_trace(
            go.Histogram(
                x=series, nbinsx=30,
                marker_color='#00CC96', opacity=0.8,
                name=column
            ),
            row=1, col=3
        )

        fig.update_layout(
            title=f'Univariate Analysis — {column}',
            template='plotly_white',
            height=400,
            showlegend=False
        )
        fig.show()

    def plot_all_univariate(self) -> None:
        """
        Run plot_univariate() for every numeric column in the DataFrame.
        """
        if not self._check_data():
            return
        for col in self.numeric_columns:
            self.plot_univariate(col)

    def plot_relationship(self, col_a: str, col_b: str) -> None:
        """
        Automatically choose and render the most appropriate chart
        based on the data types of the two columns provided.

        - Num  × Num  → Scatter with OLS trendline
        - Cat  × Num  → Box plot with jittered data points
        - Cat  × Cat  → Grouped bar chart of value-count combinations

        Parameters
        ----------
        col_a : str
            Name of the first column.
        col_b : str
            Name of the second column.
        """
        if not self._check_data():
            return
        for c in [col_a, col_b]:
            if c not in self.df.columns:
                print(f'Column "{c}" not found.')
                return

        a_num = col_a in self.numeric_columns
        b_num = col_b in self.numeric_columns

        # --- Num × Num ---
        if a_num and b_num:
            tmp = self.df[[col_a, col_b]].dropna()
            slope, intercept, r, p, _ = stats.linregress(tmp[col_a], tmp[col_b])
            x_line = np.linspace(tmp[col_a].min(), tmp[col_a].max(), 200)
            y_line = slope * x_line + intercept

            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=tmp[col_a], y=tmp[col_b],
                mode='markers',
                marker=dict(color='#636EFA', size=5, opacity=0.6),
                name='Data'
            ))
            fig.add_trace(go.Scatter(
                x=x_line, y=y_line,
                mode='lines',
                line=dict(color='#EF553B', dash='dash', width=2),
                name=f'OLS (r={r:.3f})'
            ))
            fig.update_layout(
                title=f'{col_a} vs {col_b} (Pearson r = {r:.3f}, p = {p:.4f})',
                xaxis_title=col_a, yaxis_title=col_b,
                template='plotly_white', height=450
            )
            fig.show()

        # --- Cat × Num or Num × Cat ---
        elif a_num != b_num:
            cat_col = col_b if a_num else col_a
            num_col = col_a if a_num else col_b
            tmp = self.df[[cat_col, num_col]].dropna()

            fig = px.box(
                tmp, x=cat_col, y=num_col,
                points='all',
                color=cat_col,
                title=f'{num_col} by {cat_col}',
                template='plotly_white'
            )
            fig.update_layout(height=450, showlegend=False)
            fig.show()

        # --- Cat × Cat ---
        else:
            tmp = self.df[[col_a, col_b]].dropna()
            grouped = tmp.groupby([col_a, col_b]).size().reset_index(name='count')
            fig = px.bar(
                grouped, x=col_a, y='count', color=col_b,
                barmode='group',
                title=f'{col_a} × {col_b} — Grouped Bar Chart',
                template='plotly_white'
            )
            fig.update_layout(height=450)
            fig.show()

    def plot_categorical_frequency(self, column: str) -> None:
        """
        Plot a bar chart showing the raw count and percentage label
        for each category in a categorical column.

        Parameters
        ----------
        column : str
            Name of a categorical column.
        """
        if not self._check_data():
            return
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return

        counts = self.df[column].value_counts(dropna=False)
        pcts = (counts / counts.sum() * 100).round(1)
        labels = [f'{c}<br>({p}%)' for c, p in zip(counts.values, pcts.values)]

        fig = go.Figure(go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            text=labels,
            textposition='outside',
            marker_color='#AB63FA'
        ))
        fig.update_layout(
            title=f'Frequency Distribution — {column}',
            xaxis_title=column,
            yaxis_title='Count',
            template='plotly_white',
            height=450
        )
        fig.show()

    # ------------------------------------------------------------------ #
    #  5. DEEP STATISTICAL INSIGHTS — UNIFIED ASSOCIATION HEATMAP         #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _cramers_v(x: pd.Series, y: pd.Series) -> float:
        """
        Compute Cramér's V — a symmetric measure of association
        between two categorical variables, bounded in [0, 1].

        Parameters
        ----------
        x, y : pd.Series
            Two categorical series of equal length.

        Returns
        -------
        float
            Cramér's V statistic.
        """
        contingency = pd.crosstab(x, y)
        chi2, _, _, _ = stats.chi2_contingency(contingency)
        n = contingency.sum().sum()
        r, k = contingency.shape
        return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0.0

    @staticmethod
    def _eta_squared(cat_series: pd.Series, num_series: pd.Series) -> float:
        """
        Compute Eta squared via one-way ANOVA — measures the proportion
        of variance in a numeric variable explained by a categorical variable.

        Parameters
        ----------
        cat_series : pd.Series
            Categorical grouping variable.
        num_series : pd.Series
            Continuous variable.

        Returns
        -------
        float
            Eta squared in [0, 1].
        """
        groups = [group.dropna().values
                  for _, group in num_series.groupby(cat_series)]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            return np.nan
        grand_mean = num_series.mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        ss_total = sum((v - grand_mean) ** 2 for v in num_series.dropna())
        return ss_between / ss_total if ss_total != 0 else 0.0

    def plot_all_associations_heatmap(self) -> None:
        """
        Build and display a unified correlation/association heatmap covering
        every pair of columns in the DataFrame, using the most appropriate
        metric for each pair:

        - Numeric  × Numeric     → Pearson's r
        - Categorical × Categorical → Cramér's V
        - Numeric  × Categorical → Eta squared (ANOVA-based)

        All values are normalised to [0, 1] for a consistent colour scale.
        """
        if not self._check_data():
            return

        # Use only columns with at least 2 unique non-null values
        cols = [c for c in self.df.columns
                if self.df[c].nunique(dropna=True) >= 2]

        n = len(cols)
        matrix = np.full((n, n), np.nan)

        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    matrix[i][j] = 1.0
                    continue
                if j < i:               # mirror
                    matrix[i][j] = matrix[j][i]
                    continue

                tmp = self.df[[ci, cj]].dropna()
                if len(tmp) < 5:
                    matrix[i][j] = np.nan
                    continue

                ci_num = ci in self.numeric_columns
                cj_num = cj in self.numeric_columns

                try:
                    if ci_num and cj_num:
                        r, _ = stats.pearsonr(tmp[ci], tmp[cj])
                        matrix[i][j] = r
                    elif not ci_num and not cj_num:
                        matrix[i][j] = self._cramers_v(tmp[ci], tmp[cj])
                    else:
                        cat_c = ci if not ci_num else cj
                        num_c = ci if ci_num else cj
                        matrix[i][j] = self._eta_squared(tmp[cat_c], tmp[num_c])
                except Exception:
                    matrix[i][j] = np.nan

        # Build annotation text
        annotations = np.where(
            np.isnan(matrix), '',
            np.vectorize(lambda v: f'{v:.2f}')(matrix)
        )

        fig = go.Figure(go.Heatmap(
            z=matrix,
            x=cols,
            y=cols,
            text=annotations,
            texttemplate='%{text}',
            colorscale='RdBu',
            zmid=0,
            zmin=-1, zmax=1,
            colorbar=dict(title='Association')
        ))
        fig.update_layout(
            title=(
                'Unified Association Heatmap\n'
                '(Pearson r | Cramér\'s V | Eta²)'
            ),
            template='plotly_white',
            height=max(500, 60 * n),
            width=max(600, 80 * n),
            xaxis=dict(tickangle=45)
        )
        fig.show()


print('DataInspector class defined.')

---
## Section 4 — Full Demonstration: Titanic Dataset

We now walk through the complete pipeline:

> **Load → Summarise → Impute → Deduplicate → Handle Outliers → Normalise → Visualise Associations**

### 4.1 — Load the Titanic Dataset

In [ ]:
inspector = DataInspector()

# Load Titanic directly from a public URL (no file upload needed for demo)
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
inspector.load_from_url(TITANIC_URL)

### 4.2 — Structural Summary

In [ ]:
inspector.data_summary()

### 4.3 — Remove Duplicates

In [ ]:
inspector.remove_duplicates()

### 4.4 — Drop High-Missingness & Irrelevant Columns

The `Cabin` column has ~77 % missing values and offers limited modelling value. `Name` and `Ticket` are identifiers.

In [ ]:
inspector.delete_columns('Cabin,Name,Ticket')

### 4.5 — Intelligent Imputation

* `Age` — numeric: impute with **median** (robust to skew)
* `Embarked` — categorical: impute with **mode**

In [ ]:
# Impute numeric columns with median
inspector.handle_missing_values(strategy='median', columns=['Age'])

# Impute categorical columns with mode
inspector.handle_missing_values(strategy='mode', columns=['Embarked'])

# Confirm no remaining nulls
print('\nRemaining null counts:')
print(inspector.df.isnull().sum())

### 4.6 — IQR-Based Outlier Detection

In [ ]:
# Flag outliers (adds <col>_outlier boolean columns)
outlier_summary = inspector.handle_outliers(
    columns=['Age', 'Fare'],
    action='flag',
    iqr_multiplier=1.5
)

In [ ]:
# Optionally: remove Fare outliers (extreme ticket prices)
inspector.handle_outliers(columns=['Fare'], action='remove', iqr_multiplier=3.0)

# Clean up flag columns before modelling
flag_cols = [c for c in inspector.df.columns if c.endswith('_outlier')]
if flag_cols:
    inspector.df.drop(columns=flag_cols, inplace=True)
    print(f'Removed flag columns: {flag_cols}')

### 4.7 — Univariate Visualisations

In [ ]:
inspector.plot_univariate('Age')

In [ ]:
inspector.plot_univariate('Fare')

### 4.8 — Categorical Frequency Charts

In [ ]:
inspector.plot_categorical_frequency('Sex')

In [ ]:
inspector.plot_categorical_frequency('Pclass')

In [ ]:
inspector.plot_categorical_frequency('Embarked')

### 4.9 — Smart Relationship Plots

In [ ]:
# Numeric × Numeric — Age vs Fare (OLS trendline)
inspector.plot_relationship('Age', 'Fare')

In [ ]:
# Categorical × Numeric — Survival status vs Fare (Box + jitter)
inspector.plot_relationship('Survived', 'Fare')

In [ ]:
# Categorical × Numeric — Sex vs Age
inspector.plot_relationship('Sex', 'Age')

In [ ]:
# Categorical × Categorical — Sex vs Embarked (Grouped Bar)
inspector.plot_relationship('Sex', 'Embarked')

### 4.10 — Normalisation & Feature Engineering

In [ ]:
# Preview scaled numeric features
scaled_num = inspector.extract_normalized_numeric_data(method='minmax')
print('Min-Max scaled numeric features:')
display(scaled_num.head())

In [ ]:
# Preview one-hot encoded categorical features
encoded_cat = inspector.extract_normalized_categorical_data(method='onehot')
print('One-hot encoded categorical features:')
display(encoded_cat.head())

In [ ]:
# Full model-ready merged dataset
merged_df = inspector.get_merged_normalized_dataset(
    numeric_method='standard',
    categorical_method='onehot'
)
print('Merged normalised dataset:')
display(merged_df.head())
print(f'Shape: {merged_df.shape}')

### 4.11 — Unified Association Heatmap

This heatmap visualises **every pairwise association** across all columns simultaneously:
- Numeric pairs → **Pearson r**
- Categorical pairs → **Cramér's V**
- Mixed pairs → **Eta² (ANOVA)**

In [ ]:
inspector.plot_all_associations_heatmap()

---
## Section 5 — `PlottingMethods` Standalone Demo

In [ ]:
from IPython.display import HTML

pm = PlottingMethods()

# ---- Bar Chart ----
survival_counts = inspector.df['Survived'].value_counts().rename({0: 'Did Not Survive', 1: 'Survived'})
bar_html = pm.bar_chart(survival_counts, title='Titanic Survival Counts',
                        x_label='Outcome', y_label='Passengers')
HTML(bar_html)

In [ ]:
# ---- Pie Chart ----
embarked_counts = inspector.df['Embarked'].value_counts()
pie_html = pm.pie_chart(embarked_counts, title='Passengers by Port of Embarkation')
HTML(pie_html)

In [ ]:
# ---- Histogram ----
hist_html = pm.histogram(inspector.df['Age'], title='Age Distribution',
                         x_label='Age', bins=25)
HTML(hist_html)

---
## Summary

| Requirement | Implementation |
|---|---|
| CSV upload / Colab integration | `upload_data()`, `load_from_url()`, `load_from_dataframe()` |
| Garbage string → NaN | `GARBAGE_STRINGS` list + `na_values` in `pd.read_csv` |
| Auto-type correction | `_auto_type_correction()` |
| Data summary | `data_summary()` |
| Imputation (mean/median/mode/constant) | `handle_missing_values()` |
| Duplicate removal | `remove_duplicates()` |
| IQR outlier detection | `handle_outliers()` with `flag`/`remove` |
| Targeted row/column deletion | `delete_rows()`, `delete_columns()` |
| Numeric scaling (minmax/standard/robust) | `extract_normalized_numeric_data()` |
| Categorical encoding (onehot/ordinal/uniform) | `extract_normalized_categorical_data()` |
| Merged normalised dataset | `get_merged_normalized_dataset()` |
| Univariate subplots (Violin/Scatter/Histogram) | `plot_univariate()` |
| Smart relationship plots | `plot_relationship()` |
| Categorical frequency chart | `plot_categorical_frequency()` |
| Unified association heatmap | `plot_all_associations_heatmap()` |
| Modular PlottingMethods (Bar/Pie/Histogram) | `PlottingMethods` class |
| Docstrings on every method | ✅ |
| Graceful empty/None handling | ✅ (`_check_data()` guard) |
| Real-world test (Titanic) | ✅ Full pipeline in Section 4 |